# Matching using Python

## Purpose
The purpose of this code is to analyse the impact of coupon treatment on the amount spent by customers in a retail setting on a given day. We will use a dataset that contains information about customer purchases, including whether they received a coupon and how much they spent.

## Linear Regression

In [1]:
import pandas as pd
import statsmodels.api as sm
df = pd.read_csv('../data/coupon.csv')
X = df.loc[:, df.columns != 'dailyspending']
Y = df['dailyspending']
X = sm.add_constant(X)
results = sm.OLS(Y, X).fit(cov_type='HC0')
print(results.summary())

                            OLS Regression Results                            
Dep. Variable:          dailyspending   R-squared:                       0.076
Model:                            OLS   Adj. R-squared:                  0.070
Method:                 Least Squares   F-statistic:                     12.09
Date:                Sun, 22 Feb 2026   Prob (F-statistic):           9.26e-17
Time:                        20:32:58   Log-Likelihood:                -9201.4
No. Observations:                1293   AIC:                         1.842e+04
Df Residuals:                    1284   BIC:                         1.847e+04
Df Model:                           8                                         
Covariance Type:                  HC0                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [2]:
#!pip install causalinference
from causalinference import CausalModel
import numpy as np
Y = np.asarray(df['dailyspending'])
D = np.asarray(df['coupons'])
X = np.asarray(df.drop(['dailyspending', 'coupons'], axis=1))
model = CausalModel(Y, D, X)
model.est_via_matching(weights='inv', matches=1)
print(model.estimates)


Treatment Effect Estimates: Matching

                     Est.       S.e.          z      P>|z|      [95% Conf. int.]
--------------------------------------------------------------------------------
           ATE     60.074     40.930      1.468      0.142    -20.149    140.297
           ATC     57.134     47.443      1.204      0.228    -35.854    150.122
           ATT     69.477     43.588      1.594      0.111    -15.956    154.910



`ATE (Average Treatment Effect)`: This is the estimated effect of the treatment (e.g., receiving a coupon) on the entire population. Here, the estimate is 60.074, meaning that, on average, the treatment increases daily spending by about 60 units. However, the p-value (0.142) is greater than 0.05, so this effect is not statistically significant.
`ATC (Average Treatment effect on the Controls)`: This is the estimated effect if the control group (those who did not receive the treatment) had received it. The estimate is 57.134, also not statistically significant (p = 0.228).
`ATT (Average Treatment effect on the Treated)`: This is the estimated effect for those who actually received the treatment. The estimate is 69.477, again not statistically significant (p = 0.111).

## Inverse Probability Weighting

In [3]:
#!pip install dowhy
import dowhy as dw
df = pd.read_csv('../data/coupon.csv')
X = df.drop(['dailyspending', 'coupons'], axis=1)

model = dw.CausalModel(
    data=df,
    treatment='coupons',
    outcome='dailyspending',
    common_causes=list(X.columns))

identified_estimand=model.identify_effect()

estimate = model.estimate_effect(
    identified_estimand,
    method_name="backdoor.propensity_score_weighting",
    target_units="ate",
    method_params={"weighting_scheme": "ips_weight"}
    )

print(estimate.value)
print(estimate.test_stat_significance())

c:\Users\X1 User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


76.42322609348139
{'p_value': np.float64(0.0040000000000000036)}


## Doubly Robust Estimation

In [4]:
import doubleml as dml
from sklearn import linear_model
dml_data = dml.DoubleMLData(df, 
                            y_col='dailyspending', 
                            d_cols='coupons', 
                            x_cols=list(X.columns.values))

ml_l = linear_model.LinearRegression() # model outcome
ml_m = linear_model.LogisticRegression() # model treatment
dr = dml.DoubleMLPLR(dml_data, ml_l, ml_m).fit()
print(dr.summary)

              coef   std err         t     P>|t|      2.5 %      97.5 %
coupons  77.315911  23.18299  3.335028  0.000853  31.878085  122.753736


c:\Users\X1 User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\X1 User\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.or